In [1]:
!pip install pyvi gensim polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.2 MB/s eta 0:00:00


In [2]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!unzip /content/drive/MyDrive/Database/NLP/Embedding_Weather_ForeCast.zip


Archive:  /content/drive/MyDrive/Database/NLP/Embedding_Weather_ForeCast.zip
  inflating: Dataset_articles_NoID.csv  


In [4]:
import polars as pl
import torch
import uuid
!pip install sentence_transformers


In [5]:
df = (pl.read_csv('/content/Dataset_articles_NoID.csv'))

In [6]:
display(df)

URL,Title,Summary,Contents,Date,Author(s),Category,Tags
str,str,str,str,str,str,str,str
"""https://laodong.vn/bat-dong-sa…","""Thông tin “Ngọc Trinh mua đất …","""Lâm Đồng - Lãnh đạo thành phố …","""Những ngày vừa qua, trên trang…","""Thứ sáu, 20/05/2022 08:56 (GMT…","""Phương Nhiên""","""Bất động sản""","""['Lâm Đồng', 'Ngọc Trinh', 'Ch…"
"""https://laodong.vn/bat-dong-sa…","""Lỗ hổng trong việc thẩm tra nă…","""TPHCM - Việc không thể cưỡng c…","""Theo thông tin từ Cục Thuế TP.…","""Thứ sáu, 20/05/2022 08:10 (GMT…","""Gia Miêu""","""Bất động sản""","""['Thủ Thiêm', 'Đấu giá đất']"""
"""https://laodong.vn/bat-dong-sa…","""Sớm hoàn thiện các dự án nhà ở…","""Hiện trên địa bàn tỉnh Ninh Bì…","""CNLĐ mong muốn sớm được tiếp c…","""Thứ sáu, 20/05/2022 07:47 (GMT…","""NGUYỄN TRƯỜNG""","""Bất động sản""","""['Dự án', 'Nhà ở xã hội', 'Dự …"
"""https://laodong.vn/bat-dong-sa…","""Chi tiết hồ sơ hoàn công nhà ở…","""Hoàn công nhà ở với ý nghĩa là…","""Hoàn công nhà ở là một thủ tục…","""Thứ sáu, 20/05/2022 06:44 (GMT…","""Kim Nhung (T/H)""","""Bất động sản""","""['Giấy phép xây dựng', 'Hồ sơ …"
"""https://laodong.vn/bat-dong-sa…","""Khởi tạo không gian sống đẳng …","""Có rất nhiều lý do khiến những…","""Đi dọc đường Lê Văn Lương kéo …","""Thứ năm, 19/05/2022 15:30 (GMT…","""Huyền Nguyễn""","""Bất động sản""","""['An Quý Villa']"""
…,…,…,…,…,…,…,…
"""https://laodong.vn/tlv-canh-do…","""Bà lão mù nuôi con, cháu bị ản…","""Bà Dương Thị Tuyết ở thị trấn …",null,"""Thứ tư, 14/08/2013 06:57 (GMT+…","""Lâm Hưng Thơ""","""Tấm Lòng Vàng""","""['Chất độc']"""
"""https://laodong.vn/ho-tro-bao-…","""Trao 100 triệu đồng cho ngư dâ…","""Ngày 25.7, Đại diện Chương trì…",null,"""Thứ sáu, 26/07/2013 09:03 (GMT…","""Lưu Phong""","""Tin hoạt động""","""['Phú Yên']"""
"""https://laodong.vn/giup-do-cac…","""Trao 2 “Mái ấm công đoàn” tới …","""Ngày 17.7, lãnh đạo Quỹ TLV La…",null,"""Thứ năm, 18/07/2013 07:37 (GMT…","""Bảo Duy""","""Tin hoạt động""","""['Mái ấm Công đoàn', 'Bắc Gian…"


In [7]:
print(df.filter(pl.col('Contents') != 'null').height)


307763


In [8]:
df_clean = (
    df
    .filter(pl.col('Contents') != 'null')
)
df = (
    df_clean
    .with_row_index(name="id")
    .select(
        pl.format("doc_{}", pl.col("id").alias("id")),
        pl.col("Contents").alias("content")
    )
)

In [9]:
import json
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader

In [10]:
model_name = "bkai-foundation-models/vietnamese-bi-encoder"
print(f"Downloading {model_name}")
model = SentenceTransformer(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

In [19]:
texts = df.get_column("content").to_list()
ids = df.get_column("id").to_list()
texts = texts[:1000]
print(len(texts))

1000


In [22]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
model_name = "bkai-foundation-models/vietnamese-bi-encoder"
print(f"Đang tải model {model_name}...")
model = SentenceTransformer(model_name)
model.max_seq_length = 256
train_examples = []
for text in texts:
    if isinstance(text, str) and len(text.strip()) > 0:
        train_examples.append(InputExample(texts=[text, text]))
train_dataloader = DataLoader(train_examples, shuffle = True, batch_size = 16)
train_loss = losses.MultipleNegativesRankingLoss(model)
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=100,
    show_progress_bar=True,
    output_path="./my_trained_model"
)

Đang tải model bkai-foundation-models/vietnamese-bi-encoder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
model = SentenceTransformer("./my_trained_model")
corpus_embeddings = model.encode(texts, batch_size=32, convert_to_tensor=True, show_progress_bar=True)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [28]:
from sentence_transformers import util
def query(query, top_k = 5):
  query_embedding = model.encode(query, convert_to_tensor=True)
  hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=top_k)
  top_results = hits[0]
  for i, hit in enumerate(top_results):
        vitri_bai_viet = hit['corpus_id']      # Vị trí bài viết trong danh sách texts
        diem_tuong_dong = hit['score']         # Điểm càng gần 1.0 thì càng giống
        noi_dung = texts[vitri_bai_viet]

        print(f" Top {i+1} | Độ trùng khớp: {diem_tuong_dong:.4f}")
        print(f"Nội dung: {noi_dung[:250]}...\n")


 Top 1 | Độ trùng khớp: 0.2855
Nội dung: Nguyên Vũ  là ca sĩ, diễn viên và MC nổi tiếng. Anh tham gia nghệ thuật từ năm 1990, từ khi học trung học phổ thông đã dự liên hoan ca nhạc và từng đoạt giải A năm 1993. Đến năm 1995, anh đã trở thành ca sĩ độc quyền cho trung tâm băng đĩa nhạc Vafac...

 Top 2 | Độ trùng khớp: 0.2147
Nội dung: Nguyễn Văn Chung là một nhạc sĩ nổi tiếng với nhiều bản hit hay. Anh cũng là người ghi tên mình vào sách kỷ lục khi trở thành “Nhạc sĩ trẻ sáng tác nhiều bài hát thiếu nhi nhất Việt Nam” - với khoảng 300 ca khúc. Mới đây, nam nhạc sĩ bất ngờ khoe khô...

 Top 3 | Độ trùng khớp: 0.2076
Nội dung: Dương Triệu Vũ là nam ca sĩ nổi tiếng với hàng loạt bản hit được giới trẻ yêu mến. Trong sự nghiệp của mình, nam ca sĩ từng được biết đến là một trong những ngôi sao song ca ăn ý với Đàm Vĩnh Hưng trên sân khấu. Anh cũng là một trong những ngôi sao c...

 Top 4 | Độ trùng khớp: 0.2039
Nội dung: Nam MC, diễn viên, ca sĩ Ngô Kiến Huy bắt đầu hoạt động showbiz từ năm 

In [33]:

query("Ca sĩ", top_k=5)
print("\n")
query("Pháp Luật", top_k=5)

 Top 1 | Độ trùng khớp: 0.2855
Nội dung: Nguyên Vũ  là ca sĩ, diễn viên và MC nổi tiếng. Anh tham gia nghệ thuật từ năm 1990, từ khi học trung học phổ thông đã dự liên hoan ca nhạc và từng đoạt giải A năm 1993. Đến năm 1995, anh đã trở thành ca sĩ độc quyền cho trung tâm băng đĩa nhạc Vafac...

 Top 2 | Độ trùng khớp: 0.2147
Nội dung: Nguyễn Văn Chung là một nhạc sĩ nổi tiếng với nhiều bản hit hay. Anh cũng là người ghi tên mình vào sách kỷ lục khi trở thành “Nhạc sĩ trẻ sáng tác nhiều bài hát thiếu nhi nhất Việt Nam” - với khoảng 300 ca khúc. Mới đây, nam nhạc sĩ bất ngờ khoe khô...

 Top 3 | Độ trùng khớp: 0.2076
Nội dung: Dương Triệu Vũ là nam ca sĩ nổi tiếng với hàng loạt bản hit được giới trẻ yêu mến. Trong sự nghiệp của mình, nam ca sĩ từng được biết đến là một trong những ngôi sao song ca ăn ý với Đàm Vĩnh Hưng trên sân khấu. Anh cũng là một trong những ngôi sao c...

 Top 4 | Độ trùng khớp: 0.2039
Nội dung: Nam MC, diễn viên, ca sĩ Ngô Kiến Huy bắt đầu hoạt động showbiz từ năm 